In [ ]:


import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [ ]:
import pandas as pd


single_df = pd.read_json("/run/media/victor/pessoal/mestrado/dataset/arch/pubmed_set/pubmed_set/captions.json", orient="index")

In [ ]:
single_df

In [ ]:
single_df["image_path"] = "/run/media/victor/pessoal/mestrado/dataset/arch/pubmed_set/pubmed_set/images/" + single_df["uuid"] + ".jpg"

In [ ]:
single_df

In [ ]:
single_df = single_df[~single_df["caption"].str.lower().str.strip().duplicated()].copy()

In [ ]:
single_df

In [ ]:
import os
import glob

folder_path = '/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path_double'


images_paths = glob.glob(os.path.join(folder_path, '**', '*.*'), recursive=True)
images_paths = [path for path in images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
len(images_paths)

In [ ]:
# images_paths = [path for path in images_paths if "removed" not in path]

In [ ]:
len(images_paths)

In [ ]:
new_ids = [path.split("/")[-1].replace(".jpg", "") for path in images_paths]

In [ ]:
import pandas as pd

df = pd.read_csv("/run/media/victor/pessoal/mestrado/codigo/database_scripts/V2_leiomioma_animal_clean.csv")

In [ ]:
df["new_id"] = df["article_id"] + "-" + df["id"]

In [ ]:
multi_df = df[df["new_id"].isin(new_ids)].copy()

In [ ]:
multi_df

In [ ]:
multi_df = multi_df[~multi_df["caption"].str.lower().str.strip().duplicated()]

In [ ]:
# single_df = single_df.sample(n=len(multi_df), random_state = 42).copy()

In [ ]:
len(pd.concat([single_df, multi_df])["caption"].str.lower().drop_duplicates())

In [ ]:
single_df[single_df["caption"].str.contains("×")]

In [ ]:
multi_df[multi_df["caption"].str.contains("×")]

In [ ]:
multi_df["caption"] = multi_df["caption"].str.replace("×", "x")

In [ ]:
multi_df[multi_df["caption"].str.contains("×")]

In [ ]:
single_df["label"] = 0

In [ ]:
multi_df["label"] = 1

In [ ]:
data_df = pd.concat([single_df[["caption", "label"]], multi_df[["caption", "label"]]])

In [ ]:
from torch.utils.data import Dataset, DataLoader


class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.dataframe = dataframe
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        features = row["caption"]
        label = row["label"]
        return {**self.tokenizer(features, truncation=True), "labels" : label }

    def __len__(self):
        return len(self.dataframe)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MODEL_NAME = "distilbert/distilroberta-base"  


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
data = CustomDataset(data_df, tokenizer)

In [ ]:
# from torch.utils.data import random_split

# train_data, test_data = random_split(data, [8/10, 2/10], generator=generator)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import torch

targets = data.dataframe["label"].values

train_idx, valid_idx= train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    shuffle=True,
    stratify=targets,
    random_state=SEED
)


In [ ]:
train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, valid_idx)

In [ ]:
train_data.dataset.dataframe.iloc[train_data.indices]["label"].value_counts()

In [ ]:
test_data.dataset.dataframe.iloc[test_data.indices]["label"].value_counts()

In [ ]:
import pandas as pd

# Get the captions and labels for train set
train_captions = [data.dataframe.iloc[idx]["caption"] for idx in train_data.indices]
train_labels = [data.dataframe.iloc[idx]["label"] for idx in train_data.indices]

# Get the captions and labels for test set
test_captions = [data.dataframe.iloc[idx]["caption"] for idx in test_data.indices]
test_labels = [data.dataframe.iloc[idx]["label"] for idx in test_data.indices]

# Create DataFrames

train_df = pd.DataFrame({
    'caption': train_captions,
    'label': train_labels
})

test_df = pd.DataFrame({
    'caption': test_captions,
    'label': test_labels
})

# Save to CSV
# train_df.to_csv('train_roberta_data.csv', index=False)
# test_df.to_csv('test_roberta_data.csv', index=False)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
id2label = {0: "SINGLE", 1: "MULTI"}
label2id = {"SINGLE": 0, "MULTI": 1}

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["caption"], truncation=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

In [ ]:

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)


In [ ]:
from transformers import EarlyStoppingCallback
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint



training_args = TrainingArguments(
    output_dir="V2_multi-figures-classifier",
    learning_rate=2e-6,
    per_device_train_batch_size=34,
    per_device_eval_batch_size=34,
    num_train_epochs=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,  
    load_best_model_at_end=True,  
    metric_for_best_model="eval_loss",  
    greater_is_better=False,
    seed=SEED,
    data_seed=SEED,
    # max_grad_norm=1.0,
    # gradient_accumulation_steps=4,
    logging_dir="./logs",
     report_to=["tensorboard"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
save_path = "/run/media/victor/pessoal/mestrado/codigo/model/V2_subfigures_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
save_path = "/run/media/victor/pessoal/mestrado/codigo/model/V2_subfigures_model"

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification", 
    model=save_path,
    truncation=True,
    padding=True,
    max_length=512
)

classifier("Immunohistochemical analysis showed neoplastic endometrial stromal cells immunoreactive for CD10 (IHC, ×40)")

In [ ]:

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


test_captions = [data.dataframe.iloc[idx]["caption"] for idx in test_data.indices]
true_labels = [data.dataframe.iloc[idx]["label"] for idx in test_data.indices]

predictions = classifier(test_captions)



predicted_labels = [label2id[pred["label"]] for pred in predictions]


cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["SINGLE", "MULTI"])


disp.plot(cmap="Blues")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


test_captions = [data.dataframe.iloc[idx]["caption"] for idx in test_data.indices]
true_labels = [data.dataframe.iloc[idx]["label"] for idx in test_data.indices]

predictions = classifier(test_captions)

predicted_labels = [0 if pred["score"] >= 0.8 and label2id[pred["label"]] ==0 else 1 for pred in predictions]


cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["SINGLE", "MULTI"])


disp.plot(cmap="Blues")